<a href="https://colab.research.google.com/github/evakonstantinova/EfficientNet-B0/blob/main/HQNN_6Qubit_3Layer_XZ_NoTanhPi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pennylane kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 93.2 MB/s eta 0:00:00


In [ ]:
import os
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models, transforms

from PIL import Image

import pennylane as qml
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("PyTorch:", torch.__version__)
print("PennyLane:", qml.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
PennyLane: 0.45.1
CUDA available: True
GPU: Tesla T4


In [ ]:
SEED = 42

def set_global_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_global_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random seed fixed:", SEED)

Random seed fixed: 42


In [ ]:
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one .pth checkpoint.")

uploaded_filename = next(iter(uploaded))

HQNN_CHECKPOINT = f"/content/{uploaded_filename}"

print("Checkpoint uploaded:")
print(HQNN_CHECKPOINT)

Saving BEST_HQNN_3_LAYER_XZ_NO_TANH_PI.pth to BEST_HQNN_3_LAYER_XZ_NO_TANH_PI.pth
Checkpoint uploaded:
/content/BEST_HQNN_3_LAYER_XZ_NO_TANH_PI.pth


In [ ]:
saved_state = torch.load(
    HQNN_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint loaded successfully.")

Checkpoint loaded successfully.


In [ ]:
saved_state = torch.load(
    HQNN_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint loaded successfully.")

print(
    "Feature reduction:",
    saved_state["feature_reduction.weight"].shape
)

print(
    "Quantum weights:",
    saved_state["quantum_layer.weights"].shape
)

print(
    "Classifier:",
    saved_state["classifier.weight"].shape
)

Checkpoint loaded successfully.
Feature reduction: torch.Size([4, 1280])
Quantum weights: torch.Size([3, 4, 3])
Classifier: torch.Size([4, 8])


In [ ]:
from torchvision import models

# Create EfficientNet-B0 architecture
efficientnet = models.efficientnet_b0(
    weights=None
)

FEATURE_DIM = efficientnet.classifier[1].in_features

assert FEATURE_DIM == 1280


# Extract ONLY the trained EfficientNet feature extractor
feature_state = {
    key.replace("features.", "", 1): value
    for key, value in saved_state.items()
    if key.startswith("features.")
}


# Load trained backbone weights
efficientnet.features.load_state_dict(
    feature_state,
    strict=True
)


# Freeze EfficientNet
for param in efficientnet.features.parameters():
    param.requires_grad = False


print("EfficientNet-B0 backbone restored successfully.")
print("Feature dimension:", FEATURE_DIM)

print(
    "Trainable EfficientNet parameters:",
    sum(
        p.numel()
        for p in efficientnet.features.parameters()
        if p.requires_grad
    )
)

EfficientNet-B0 backbone restored successfully.
Feature dimension: 1280
Trainable EfficientNet parameters: 0


In [ ]:
# ==========================================
# SAME DATASET AND SAME 70/15/15 SPLIT
# ==========================================

from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
import kagglehub

path = kagglehub.dataset_download(
    "masoudnickparvar/brain-tumor-mri-dataset"
)

classes = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]

all_files = []
all_labels = []

for class_name in classes:

    for folder in ["Training", "Testing"]:

        class_path = Path(path) / folder / class_name

        for file_path in class_path.iterdir():

            if file_path.is_file():

                all_files.append(str(file_path))
                all_labels.append(class_name)


train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files,
    all_labels,
    test_size=0.30,
    random_state=42,
    stratify=all_labels
)

val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files,
    temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

print("Total images:", len(all_files))
print("Training:", len(train_files), Counter(train_labels))
print("Validation:", len(val_files), Counter(val_labels))
print("Testing:", len(test_files), Counter(test_labels))

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Total images: 7200
Training: 5040 Counter({'notumor': 1260, 'pituitary': 1260, 'glioma': 1260, 'meningioma': 1260})
Validation: 1080 Counter({'notumor': 270, 'glioma': 270, 'meningioma': 270, 'pituitary': 270})
Testing: 1080 Counter({'meningioma': 270, 'notumor': 270, 'glioma': 270, 'pituitary': 270})


In [ ]:
# ==========================================
# 6-QUBIT, 3-LAYER, X + Z READOUT CIRCUIT
# ==========================================

import pennylane as qml

N_QUBITS = 6
N_Q_LAYERS = 3
N_QUANTUM_OUTPUTS = 12


quantum_device_6q = qml.device(
    "default.qubit",
    wires=N_QUBITS
)


@qml.qnode(
    quantum_device_6q,
    interface="torch",
    diff_method="backprop"
)
def quantum_circuit_6q(inputs, weights):

    # 6 reduced classical features -> 6 qubits
    # NO tanh × pi
    qml.AngleEmbedding(
        inputs,
        wires=range(N_QUBITS),
        rotation="Y"
    )

    # Same depth as our best 4-qubit model
    qml.StronglyEntanglingLayers(
        weights,
        wires=range(N_QUBITS)
    )

    # X + Z readout
    x_measurements = [
        qml.expval(qml.PauliX(i))
        for i in range(N_QUBITS)
    ]

    z_measurements = [
        qml.expval(qml.PauliZ(i))
        for i in range(N_QUBITS)
    ]

    return x_measurements + z_measurements


weight_shapes_6q = {
    "weights": (
        N_Q_LAYERS,
        N_QUBITS,
        3
    )
}


set_global_seed(SEED)

quantum_layer_6q = qml.qnn.TorchLayer(
    quantum_circuit_6q,
    weight_shapes_6q
)


print("6-qubit quantum circuit created successfully.")
print("Qubits:", N_QUBITS)
print("Quantum layers:", N_Q_LAYERS)
print("Quantum parameters:", 54)
print("Quantum outputs:", N_QUANTUM_OUTPUTS)
print("Readout: X + Z")
print("tanh × pi: REMOVED")

6-qubit quantum circuit created successfully.
Qubits: 6
Quantum layers: 3
Quantum parameters: 54
Quantum outputs: 12
Readout: X + Z
tanh × pi: REMOVED


In [ ]:
# ==========================================
# CREATE 6-QUBIT HQNN MODEL
# ==========================================

class HQNN6Qubit(nn.Module):

    def __init__(
        self,
        trained_efficientnet,
        quantum_layer
    ):
        super().__init__()

        # SAME TRAINED EFFICIENTNET BACKBONE
        self.features = trained_efficientnet.features
        self.avgpool = trained_efficientnet.avgpool

        # KEEP BACKBONE FROZEN
        for param in self.features.parameters():
            param.requires_grad = False

        # NEW 1280 -> 6 FEATURE REDUCTION
        self.feature_reduction = nn.Linear(
            FEATURE_DIM,
            N_QUBITS
        )

        # NEW 6-QUBIT QUANTUM LAYER
        self.quantum_layer = quantum_layer

        # 12 X+Z MEASUREMENTS -> 4 CLASSES
        self.classifier = nn.Linear(
            N_QUANTUM_OUTPUTS,
            4
        )

    def forward(self, x):

        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        # 1280 -> 6
        x = self.feature_reduction(x)

        # NO tanh × pi

        x = x.cpu()

        # 6 qubits -> 12 X+Z expectation values
        x = self.quantum_layer(x)

        classifier_device = next(
            self.classifier.parameters()
        ).device

        x = x.to(classifier_device)

        # 12 -> 4
        x = self.classifier(x)

        return x


set_global_seed(SEED)

hqnn_6q = HQNN6Qubit(
    efficientnet,
    quantum_layer_6q
)

print(hqnn_6q)

HQNN6Qubit(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivatio

In [ ]:
# ==========================================
# VERIFY TRAINABLE PARAMETERS
# ==========================================

backbone_trainable = sum(
    p.numel()
    for p in hqnn_6q.features.parameters()
    if p.requires_grad
)

total_trainable = sum(
    p.numel()
    for p in hqnn_6q.parameters()
    if p.requires_grad
)

print(
    "Trainable EfficientNet parameters:",
    backbone_trainable
)

print(
    "Total trainable parameters:",
    total_trainable
)

print("\nTrainable components:")

for name, parameter in hqnn_6q.named_parameters():

    if parameter.requires_grad:

        print(
            name,
            tuple(parameter.shape),
            parameter.numel()
        )

Trainable EfficientNet parameters: 0
Total trainable parameters: 7792

Trainable components:
feature_reduction.weight (6, 1280) 7680
feature_reduction.bias (6,) 6
quantum_layer.weights (3, 6, 3) 54
classifier.weight (4, 12) 48
classifier.bias (4,) 4


In [ ]:
# ==========================================
# TRANSFORMS + DATASETS + DATALOADERS
# ==========================================

class_to_idx = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class BrainTumorDataset(Dataset):

    def __init__(self, files, labels, transform=None):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        image = Image.open(
            self.files[idx]
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[
            self.labels[idx]
        ]

        return image, label


train_dataset = BrainTumorDataset(
    train_files,
    train_labels,
    transform=train_transform
)

val_dataset = BrainTumorDataset(
    val_files,
    val_labels,
    transform=val_test_transform
)

test_dataset = BrainTumorDataset(
    test_files,
    test_labels,
    transform=val_test_transform
)


BATCH_SIZE = 32


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Testing images:", len(test_dataset))
print("Batch size:", BATCH_SIZE)

Training images: 5040
Validation images: 1080
Testing images: 1080
Batch size: 32


In [ ]:
# ==========================================
# GPU FEATURE EXTRACTION
# ==========================================

feature_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Feature extraction device:", feature_device)


hqnn_6q.features = hqnn_6q.features.to(feature_device)
hqnn_6q.avgpool = hqnn_6q.avgpool.to(feature_device)

hqnn_6q.features.eval()
hqnn_6q.avgpool.eval()


def extract_efficientnet_features(data_loader):

    all_features = []
    all_labels = []

    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(feature_device)

            features = hqnn_6q.features(images)
            features = hqnn_6q.avgpool(features)
            features = torch.flatten(features, 1)

            all_features.append(
                features.cpu()
            )

            all_labels.append(
                labels.cpu()
            )

    return (
        torch.cat(all_features, dim=0),
        torch.cat(all_labels, dim=0)
    )


print("Feature extraction function ready.")

Feature extraction device: cuda
Feature extraction function ready.


In [ ]:
# ==========================================
# EXTRACT VALIDATION FEATURES
# ==========================================

print("Extracting validation features...")

val_features, val_labels = (
    extract_efficientnet_features(
        val_loader
    )
)

print("Validation feature extraction completed.")
print("Validation features shape:", val_features.shape)
print("Validation labels shape:", val_labels.shape)

Extracting validation features...
Validation feature extraction completed.
Validation features shape: torch.Size([1080, 1280])
Validation labels shape: torch.Size([1080])


In [ ]:
# ==========================================
# TRAINING SETUP — 6-QUBIT HQNN
# ==========================================

hqnn_6q.feature_reduction = hqnn_6q.feature_reduction.cpu()
hqnn_6q.quantum_layer = hqnn_6q.quantum_layer.cpu()
hqnn_6q.classifier = hqnn_6q.classifier.cpu()


criterion = nn.CrossEntropyLoss()


trainable_parameters = (
    list(hqnn_6q.feature_reduction.parameters())
    + list(hqnn_6q.quantum_layer.parameters())
    + list(hqnn_6q.classifier.parameters())
)


optimizer = torch.optim.Adam(
    trainable_parameters,
    lr=0.001
)


NUM_EPOCHS = 10

best_val_loss = float("inf")
best_epoch = 0


print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Maximum epochs:", NUM_EPOCHS)
print("Qubits: 6")
print("Quantum layers: 3")
print("Readout: X + Z")
print("EfficientNet backbone trainable: NO")
print("tanh × pi scaling: REMOVED")

Loss function: CrossEntropyLoss
Optimizer: Adam
Learning rate: 0.001
Maximum epochs: 10
Qubits: 6
Quantum layers: 3
Readout: X + Z
EfficientNet backbone trainable: NO
tanh × pi scaling: REMOVED


In [ ]:
# ==========================================
# TRAIN 6-QUBIT HQNN
# ==========================================

train_losses = []
train_accuracies = []
train_f1_scores = []

val_losses = []
val_accuracies = []
val_f1_scores = []


training_start_time = time.time()


for epoch in range(NUM_EPOCHS):

    set_global_seed(SEED + epoch)

    print(f"\nEPOCH {epoch + 1}/{NUM_EPOCHS}")
    print("Extracting augmented training features...")


    # ------------------------------------------
    # Extract new augmented training features
    # ------------------------------------------

    train_features, train_labels = (
        extract_efficientnet_features(
            train_loader
        )
    )


    train_feature_dataset = TensorDataset(
        train_features,
        train_labels
    )


    train_feature_loader = DataLoader(
        train_feature_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )


    # ==========================================
    # TRAINING
    # ==========================================

    hqnn_6q.feature_reduction.train()
    hqnn_6q.quantum_layer.train()
    hqnn_6q.classifier.train()


    running_train_loss = 0.0

    train_predictions = []
    train_targets = []


    for features, labels in train_feature_loader:

        optimizer.zero_grad()


        # 1280 -> 6
        outputs = hqnn_6q.feature_reduction(
            features
        )


        # NO tanh × pi


        # 6-qubit quantum circuit
        outputs = hqnn_6q.quantum_layer(
            outputs
        )


        # 12 X+Z outputs -> 4 classes
        outputs = hqnn_6q.classifier(
            outputs
        )


        loss = criterion(
            outputs,
            labels
        )


        loss.backward()

        optimizer.step()


        running_train_loss += (
            loss.item()
            * features.size(0)
        )


        predictions = torch.argmax(
            outputs,
            dim=1
        )


        train_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        train_targets.extend(
            labels.cpu().numpy()
        )


    epoch_train_loss = (
        running_train_loss
        / len(train_feature_dataset)
    )


    epoch_train_accuracy = accuracy_score(
        train_targets,
        train_predictions
    )


    epoch_train_f1 = f1_score(
        train_targets,
        train_predictions,
        average="macro"
    )


    # ==========================================
    # VALIDATION
    # ==========================================

    hqnn_6q.feature_reduction.eval()
    hqnn_6q.quantum_layer.eval()
    hqnn_6q.classifier.eval()


    val_feature_dataset = TensorDataset(
        val_features,
        val_labels
    )


    val_feature_loader = DataLoader(
        val_feature_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )


    running_val_loss = 0.0

    val_predictions = []
    val_targets = []


    with torch.no_grad():

        for features, labels in val_feature_loader:

            # 1280 -> 6
            outputs = hqnn_6q.feature_reduction(
                features
            )


            # 6-qubit quantum layer
            outputs = hqnn_6q.quantum_layer(
                outputs
            )


            # 12 -> 4
            outputs = hqnn_6q.classifier(
                outputs
            )


            loss = criterion(
                outputs,
                labels
            )


            running_val_loss += (
                loss.item()
                * features.size(0)
            )


            predictions = torch.argmax(
                outputs,
                dim=1
            )


            val_predictions.extend(
                predictions.cpu().numpy()
            )

            val_targets.extend(
                labels.cpu().numpy()
            )


    epoch_val_loss = (
        running_val_loss
        / len(val_feature_dataset)
    )


    epoch_val_accuracy = accuracy_score(
        val_targets,
        val_predictions
    )


    epoch_val_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro"
    )


    # ==========================================
    # SAVE METRICS
    # ==========================================

    train_losses.append(
        epoch_train_loss
    )

    train_accuracies.append(
        epoch_train_accuracy
    )

    train_f1_scores.append(
        epoch_train_f1
    )

    val_losses.append(
        epoch_val_loss
    )

    val_accuracies.append(
        epoch_val_accuracy
    )

    val_f1_scores.append(
        epoch_val_f1
    )


    # ==========================================
    # SAVE BEST CHECKPOINT
    # ==========================================

    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss
        best_epoch = epoch + 1


        torch.save(
            hqnn_6q.state_dict(),
            "/content/BEST_HQNN_6Q_3L_XZ_NO_TANH_PI.pth"
        )


        marker = " <-- BEST CHECKPOINT"

    else:

        marker = ""


    # ==========================================
    # PRINT EPOCH RESULTS
    # ==========================================

    print(
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_accuracy:.4f} | "
        f"Train F1: {epoch_train_f1:.4f}"
    )


    print(
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_accuracy:.4f} | "
        f"Val F1: {epoch_val_f1:.4f}"
        f"{marker}"
    )


training_time = (
    time.time()
    - training_start_time
)


print("\nTRAINING COMPLETED")

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation loss:",
    round(best_val_loss, 4)
)

print(
    "Training time:",
    round(training_time / 60, 2),
    "minutes"
)


EPOCH 1/10
Extracting augmented training features...
Train Loss: 1.1495 | Train Acc: 0.7609 | Train F1: 0.7664
Val Loss: 0.9913 | Val Acc: 0.9241 | Val F1: 0.9243 <-- BEST CHECKPOINT

EPOCH 2/10
Extracting augmented training features...
Train Loss: 0.8304 | Train Acc: 0.9472 | Train F1: 0.9474
Val Loss: 0.7106 | Val Acc: 0.9370 | Val F1: 0.9370 <-- BEST CHECKPOINT

EPOCH 3/10
Extracting augmented training features...
Train Loss: 0.5618 | Train Acc: 0.9536 | Train F1: 0.9536
Val Loss: 0.4852 | Val Acc: 0.9324 | Val F1: 0.9318 <-- BEST CHECKPOINT

EPOCH 4/10
Extracting augmented training features...
Train Loss: 0.3709 | Train Acc: 0.9617 | Train F1: 0.9617
Val Loss: 0.3512 | Val Acc: 0.9389 | Val F1: 0.9388 <-- BEST CHECKPOINT

EPOCH 5/10
Extracting augmented training features...
Train Loss: 0.2545 | Train Acc: 0.9698 | Train F1: 0.9700
Val Loss: 0.2563 | Val Acc: 0.9574 | Val F1: 0.9574 <-- BEST CHECKPOINT

EPOCH 6/10
Extracting augmented training features...
Train Loss: 0.1895 | Train

In [ ]:
# ==========================================
# FINAL TEST — 6-QUBIT HQNN
# 3 LAYERS + X/Z READOUT + NO TANH × PI
# ==========================================

best_state_6q = torch.load(
    "/content/BEST_HQNN_6Q_3L_XZ_NO_TANH_PI.pth",
    map_location="cpu",
    weights_only=True
)

hqnn_6q.load_state_dict(
    best_state_6q
)

print("Best 6-qubit HQNN checkpoint loaded successfully.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)


# ==========================================
# EXTRACT TEST FEATURES
# ==========================================

print("\nExtracting test features...")

test_features, test_labels = (
    extract_efficientnet_features(
        test_loader
    )
)

print("Test feature extraction completed.")
print("Test features shape:", test_features.shape)
print("Test labels shape:", test_labels.shape)


# ==========================================
# TEST DATALOADER
# ==========================================

test_feature_dataset = TensorDataset(
    test_features,
    test_labels
)

test_feature_loader = DataLoader(
    test_feature_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# ==========================================
# EVALUATION MODE
# ==========================================

hqnn_6q.feature_reduction.eval()
hqnn_6q.quantum_layer.eval()
hqnn_6q.classifier.eval()


test_targets = []
test_predictions = []
test_probabilities = []

running_test_loss = 0.0


# ==========================================
# FINAL TEST
# ==========================================

with torch.no_grad():

    for features, labels in test_feature_loader:

        # 1280 -> 6
        outputs = hqnn_6q.feature_reduction(
            features
        )

        # NO tanh × pi

        # 6 qubits -> 12 X+Z measurements
        outputs = hqnn_6q.quantum_layer(
            outputs
        )

        # 12 -> 4 classes
        outputs = hqnn_6q.classifier(
            outputs
        )

        loss = criterion(
            outputs,
            labels
        )

        running_test_loss += (
            loss.item()
            * features.size(0)
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        test_targets.extend(
            labels.cpu().numpy()
        )

        test_predictions.extend(
            predictions.cpu().numpy()
        )

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )


# ==========================================
# METRICS
# ==========================================

test_targets = np.array(
    test_targets
)

test_predictions = np.array(
    test_predictions
)

test_probabilities = np.array(
    test_probabilities
)


test_loss = (
    running_test_loss
    / len(test_feature_dataset)
)


test_accuracy = accuracy_score(
    test_targets,
    test_predictions
)

test_precision = precision_score(
    test_targets,
    test_predictions,
    average="macro"
)

test_recall = recall_score(
    test_targets,
    test_predictions,
    average="macro"
)

test_f1 = f1_score(
    test_targets,
    test_predictions,
    average="macro"
)

test_auc = roc_auc_score(
    test_targets,
    test_probabilities,
    multi_class="ovr",
    average="macro"
)


# ==========================================
# RESULTS
# ==========================================

print("\nFINAL 6-QUBIT HQNN TEST RESULTS")

print(
    f"Test Loss:       {test_loss:.4f}"
)

print(
    f"Accuracy:        "
    f"{test_accuracy:.4f} "
    f"({test_accuracy * 100:.2f}%)"
)

print(
    f"Macro Precision: "
    f"{test_precision:.4f} "
    f"({test_precision * 100:.2f}%)"
)

print(
    f"Macro Recall:    "
    f"{test_recall:.4f} "
    f"({test_recall * 100:.2f}%)"
)

print(
    f"Macro F1:        "
    f"{test_f1:.4f} "
    f"({test_f1 * 100:.2f}%)"
)

print(
    f"Macro ROC-AUC:   "
    f"{test_auc:.4f} "
    f"({test_auc * 100:.2f}%)"
)


print("\nCLASSIFICATION REPORT\n")

print(
    classification_report(
        test_targets,
        test_predictions,
        target_names=[
            "Glioma",
            "Meningioma",
            "No Tumor",
            "Pituitary"
        ],
        digits=4
    )
)


print("\nCONFUSION MATRIX")

print(
    confusion_matrix(
        test_targets,
        test_predictions
    )
)

Best 6-qubit HQNN checkpoint loaded successfully.
Best epoch: 10
Best validation loss: 0.14187507237549182

Extracting test features...
Test feature extraction completed.
Test features shape: torch.Size([1080, 1280])
Test labels shape: torch.Size([1080])

FINAL 6-QUBIT HQNN TEST RESULTS
Test Loss:       0.1908
Accuracy:        0.9500 (95.00%)
Macro Precision: 0.9506 (95.06%)
Macro Recall:    0.9500 (95.00%)
Macro F1:        0.9497 (94.97%)
Macro ROC-AUC:   0.9928 (99.28%)

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      Glioma     0.9798    0.9000    0.9382       270
  Meningioma     0.9081    0.9148    0.9114       270
    No Tumor     0.9573    0.9963    0.9764       270
   Pituitary     0.9570    0.9889    0.9727       270

    accuracy                         0.9500      1080
   macro avg     0.9506    0.9500    0.9497      1080
weighted avg     0.9506    0.9500    0.9497      1080


CONFUSION MATRIX
[[243  21   5   1]
 [  5 247   7  11]
 [  0   